# Simple CNN Demo - Minimal Network for OpenEye

This notebook demonstrates the OpenEye workflow with the **simplest possible neural network**:
- 1 Conv2D layer (3×3 kernel)
- 1 Dense/FC layer

Perfect for understanding the basics without complexity!

## Network Architecture
```
Input (28×28×1) → Conv2D(3×3, 8 filters) → Flatten → Dense(10) → Output
```

## Steps
1. Build and train simple model
2. Quantize with TFLite
3. Load with OpenEye
4. Inspect hardware mapping

## Step 1: Build Simple Model

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow version: {tf.__version__}")

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape((x_train.shape[0], 28, 28, 1)).astype('float32') / 255
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1)).astype('float32') / 255
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print(f"Training data: {x_train.shape}")
print(f"Test data: {x_test.shape}")

In [ ]:
# Build the SIMPLEST possible CNN
model = models.Sequential([
    # Single Conv2D layer: 3×3 kernel, 8 filters, same padding
    layers.Conv2D(8, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    
    # Flatten to 1D
    layers.Flatten(),
    
    # Single Dense layer: 10 outputs (one per digit)
    layers.Dense(10, activation='softmax')
])

# Compile
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("Simple Model Architecture:")
print("=" * 60)
model.summary()
print("\nTotal layers: 3 (Conv2D + Flatten + Dense)")

## Step 2: Train the Model

We'll train for just 3 epochs - this simple model won't achieve great accuracy, but that's not the point. We're demonstrating the hardware workflow!

In [ ]:
# Train the model (quick training - just 3 epochs)
print("Training simple model...")
history = model.fit(x_train, y_train, 
                    epochs=3, 
                    batch_size=128, 
                    validation_split=0.1, 
                    verbose=1)

# Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print("Note: Accuracy is low because this is a minimal model for demo purposes!")

## Step 3: Quantize to INT8

In [ ]:
# Convert to TFLite with INT8 quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.target_spec.supported_types = [tf.int8]

def representative_dataset_gen():
    """Representative dataset for quantization"""
    for i in range(100):
        sample = np.expand_dims(x_train[i], axis=0).astype(np.float32)
        yield [sample]

converter.representative_dataset = representative_dataset_gen
tflite_model = converter.convert()

# Save
model_name = "simple_cnn_quantized"
model_path = model_name + '.tflite'
with open(model_path, 'wb') as f:
    f.write(tflite_model)

print(f"\n✅ Quantized model saved: {model_path}")

# Show size
import os
size_kb = os.path.getsize(model_path) / 1024
print(f"   Model size: {size_kb:.2f} KB")

## Step 4: Verify Quantized Model

In [ ]:
# Test quantized model
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Quantized Model Info:")
print("=" * 60)
print(f"Input:  shape={input_details[0]['shape']}, dtype={input_details[0]['dtype']}")
print(f"Output: shape={output_details[0]['shape']}, dtype={output_details[0]['dtype']}")

# Test inference
test_sample = x_test[0:1].astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_sample)
interpreter.invoke()
tflite_result = interpreter.get_tensor(output_details[0]['index'])

original_pred = np.argmax(model.predict(test_sample, verbose=0))
quantized_pred = np.argmax(tflite_result)

print(f"\nPrediction Test:")
print(f"  Original model:  {original_pred}")
print(f"  Quantized model: {quantized_pred}")
print(f"  Match: {'✅' if original_pred == quantized_pred else '❌'}")

## Step 5: Load with OpenEye Unified Loader

In [ ]:
import sys
import os

# Add OpenEye to path
openeye_base = os.path.abspath(os.path.join(os.pardir, os.pardir))
sys.path.insert(0, os.path.join(openeye_base, "src"))

from open_eye.model_loader import load_model
from open_eye.layer_adapter import adapt_layers_for_keras

# Load model
openeye_model = load_model(model_path)

print("OpenEye Model Properties:")
print("=" * 60)
print(f"Framework:    {openeye_model.framework}")
print(f"Layers:       {openeye_model.num_layers}")
print(f"Input shape:  {openeye_model.input_shape}")
print(f"Output shape: {openeye_model.output_shape}")
print(f"Quantized:    {openeye_model.is_quantized}")

print("\nLayer Structure:")
for i, layer in enumerate(openeye_model.layers):
    layer_name = layer.__class__.__name__
    in_shape = layer.input_shape if hasattr(layer, 'input_shape') else 'N/A'
    out_shape = layer.output_shape if hasattr(layer, 'output_shape') else 'N/A'
    print(f"  {i}: {layer_name:20s} {in_shape} → {out_shape}")

print("\n✅ Model loaded successfully!")

## Step 6: Adapt Layers and Inspect Details

In [ ]:
# Adapt layers for OpenEye hardware
adapted_layers = adapt_layers_for_keras(openeye_model.layers)

print("Adapted Layers for OpenEye Hardware:")
print("=" * 70)

for i, layer in enumerate(adapted_layers):
    layer_name = layer.name if hasattr(layer, 'name') else 'Unknown'
    output_shape = layer.output.shape if hasattr(layer, 'output') else 'N/A'
    
    print(f"\nLayer {i}: {layer_name}")
    print(f"  Output shape: {output_shape}")
    
    # Conv2D details
    if layer_name == 'conv2d':
        print(f"  Kernel size:  {layer.kernel_size if hasattr(layer, 'kernel_size') else 'N/A'}")
        print(f"  Filters:      {layer.filters if hasattr(layer, 'filters') else 'N/A'}")
        print(f"  Strides:      {layer.strides if hasattr(layer, 'strides') else 'N/A'}")
        print(f"  Padding:      {layer.padding if hasattr(layer, 'padding') else 'N/A'}")
        if hasattr(layer, 'kernel'):
            print(f"  Weight shape: {layer.kernel.shape}")
    
    # Dense details
    elif layer_name == 'dense':
        print(f"  Units:        {layer.units if hasattr(layer, 'units') else 'N/A'}")
        if hasattr(layer, 'kernel'):
            print(f"  Weight shape: {layer.kernel.shape}")
    
    # Flatten details
    elif layer_name == 'flatten':
        print(f"  (No parameters)")

print("\n✅ All layers adapted successfully!")

## Step 7: Hardware Mapping Preview

Let's see how the Conv2D layer would map to OpenEye hardware.

In [ ]:
# Find the Conv2D layer
conv_layer = None
for layer in adapted_layers:
    if hasattr(layer, 'name') and layer.name == 'conv2d':
        conv_layer = layer
        break

if conv_layer:
    print("Conv2D Layer Hardware Mapping:")
    print("=" * 70)
    print(f"Input:       {conv_layer.input.shape}")
    print(f"Output:      {conv_layer.output.shape}")
    print(f"Kernel:      {conv_layer.kernel_size}")
    print(f"Filters:     {conv_layer.filters}")
    print(f"Strides:     {conv_layer.strides}")
    
    # Calculate operations
    if hasattr(conv_layer, 'kernel'):
        kernel_h, kernel_w, in_channels, out_channels = conv_layer.kernel.shape
        out_h, out_w = conv_layer.output.shape[1:3]
        
        total_ops = out_h * out_w * kernel_h * kernel_w * in_channels * out_channels
        
        print(f"\nComputational Requirements:")
        print(f"  Operations: {total_ops:,} MACs")
        print(f"  Weights:    {np.prod(conv_layer.kernel.shape):,} parameters")
        
        # Simple hardware estimate (assuming 2×3 PE cluster)
        pe_count = 2 * 3  # Example: 2×3 PEs
        cycles_estimate = total_ops / pe_count
        print(f"\nHardware Estimate (2×3 PEs):")
        print(f"  Cycles: ~{cycles_estimate:,.0f}")
else:
    print("⚠️  Conv2D layer not found")

print("\n✅ Hardware mapping preview complete!")

## Summary

This notebook demonstrated the **complete OpenEye workflow** with the simplest possible CNN:

✅ **Built** minimal 2-layer network (Conv2D + Dense)  
✅ **Trained** on MNIST (3 epochs, quick demo)  
✅ **Quantized** to INT8 with TFLite  
✅ **Loaded** with OpenEye unified loader  
✅ **Adapted** layers for hardware compatibility  
✅ **Inspected** hardware mapping requirements  

## Network Architecture
```
Input (28×28×1)
    ↓
Conv2D (3×3, 8 filters, same padding, ReLU)
    ↓
Flatten (28×28×8 → 6272)
    ↓
Dense (6272 → 10, softmax)
    ↓
Output (10 classes)
```

## Key Takeaways

1. **Minimal complexity** - Only 2 learnable layers (Conv + Dense)
2. **Small model** - Just a few KB when quantized
3. **Fast training** - 3 epochs, < 1 minute
4. **Full workflow** - All OpenEye steps demonstrated
5. **Hardware ready** - Quantized and adapted for deployment

## Next Steps

- Try with different kernel sizes (5×5, 7×7)
- Add more filters to the Conv2D layer
- Run RTL simulation (see [mnist.ipynb](mnist.ipynb))
- Compare with more complex architectures
- Deploy on FPGA hardware

For more complex examples, see:
- [mnist.ipynb](mnist.ipynb) - Full MNIST workflow with deeper CNN
- [mnist_tensorflow.ipynb](mnist_tensorflow.ipynb) - Complete TensorFlow tutorial
- [unified_model_loader_demo.ipynb](unified_model_loader_demo.ipynb) - Multi-framework demo